<a href="https://colab.research.google.com/github/Sourour21/ProjectML-IDS/blob/main/_ProjectML_IDSipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q shap
import os, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, xgboost, shap
SEED = 42 # graine : mêmes résultats à chaque exécution
random.seed(SEED); np.random.seed(SEED)
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/Project1ML/Data/'
MODELES = '/content/drive/MyDrive/Project1ML/modeles/'
os.makedirs(MODELES, exist_ok=True)
# versions à recopier dans le README
print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__,
'| xgboost', xgboost.__version__, '| shap', shap.__version__)

Mounted at /content/drive
pandas 2.2.3 | scikit-learn 1.6.1 | xgboost 3.4.1 | shap 0.54.1


In [3]:
print(os.listdir(DATA))

['UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv']


Étape 1 : description des données

Le jeu UNSW-NB15 a été créé en 2015 par l’UNSW Canberra (Moustafa et Slay) à partir de trafic réseau normal et d’attaques générés dans un banc de test. On utilise les deux fichiers officiels pré-découpés : l’entraînement contient 175 341 flux et le test 82 332, chacun avec 45 colonnes.

Parmi ces colonnes, 42 sont des variables explicatives, auxquelles s’ajoutent id (numéro de ligne, sans signification), attack_cat (la famille d’attaque) et label (la cible : 0 normal, 1 attaque). Trois variables sont catégorielles (proto, service, state) ; les autres sont numériques. Aucune valeur manquante n’apparaît à ce stade.

Les attaques représentent 68,1 % de l’entraînement (31,9 % de flux normaux). Il y a neuf familles d’attaques, très inégales : Generic (40 000) et Exploits (33 393) dominent, tandis que Worms n’a que 130 flux et Shellcode 1 133.

Limite importante : les attaques sont majoritaires, alors qu’en réseau réel elles sont rares. La précision mesurée ici sera donc plus élevée qu’en production, et un modèle au hasard aurait une PR-AUC d’environ 0,68, qui sert de référence. Les familles rares (Worms, Shellcode, Backdoor) risquent aussi d’être mal détectées.

In [4]:
train_full = pd.read_csv(DATA + 'UNSW_NB15_training-set.csv')
test = pd.read_csv(DATA + 'UNSW_NB15_testing-set.csv')
print(train_full.shape, test.shape)  # attendu : (175341, 45) et (82332, 45)
display(train_full.head())
train_full.info()
display(train_full['label'].value_counts(normalize=True))
display(train_full['attack_cat'].value_counts())

(175341, 45) (82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

,proportion
label,
1,0.680622
0,0.319378


,count
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133


In [5]:
# 1. valeurs infinies et manquantes
for df in (train_full, test):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
print('Valeurs manquantes :', train_full.isna().sum().sum(), test.isna().sum().sum())
train_full = train_full.dropna()
test = test.dropna()

# 2. variables retirées : id (numéro de ligne), label (la réponse), attack_cat (la réponse détaillée)
DROP = ['id', 'label', 'attack_cat']
FEATURES = [c for c in train_full.columns if c not in DROP]
CAT = ['proto', 'service', 'state']

# 3. doublons dans l'entraînement
n0 = len(train_full)
train_full = train_full.drop_duplicates(subset=FEATURES + ['label'])
print("Doublons retirés de l'entraînement :", n0 - len(train_full))
conflits = train_full.groupby(FEATURES)['label'].nunique()
print('Flux identiques avec des étiquettes différentes :', (conflits > 1).sum())

# 4. lignes du test déjà présentes dans l'entraînement (fuite)
m = test.merge(train_full[FEATURES].drop_duplicates(), on=FEATURES, how='left', indicator=True)
print(f"Part du test déjà présente dans l'entraînement : {(m['_merge'] == 'both').mean():.1%}")
test_clean = m[m['_merge'] == 'left_only'].drop(columns='_merge')
test_clean = test_clean.drop_duplicates(subset=FEATURES + ['label'])
print('Test après nettoyage :', test_clean.shape)

Valeurs manquantes : 0 0
Doublons retirés de l'entraînement : 74072
Flux identiques avec des étiquettes différentes : 229
Part du test déjà présente dans l'entraînement : 10.4%
Test après nettoyage : (52648, 45)


In [6]:
print("Entraînement après nettoyage :", train_full.shape)
print(train_full['label'].value_counts(normalize=True).round(4))
print("\nTest après nettoyage :", test_clean.shape)
print(test_clean['label'].value_counts(normalize=True).round(4))
display(test_clean['attack_cat'].value_counts())

Entraînement après nettoyage : (101269, 45)
label
0    0.5124
1    0.4876
Name: proportion, dtype: float64

Test après nettoyage : (52648, 45)
label
0    0.6395
1    0.3605
Name: proportion, dtype: float64


,count
attack_cat,
Normal,33666
Exploits,7085
Fuzzers,4335
Generic,3381
Reconnaissance,2224
DoS,1186
Shellcode,364
Analysis,294
Backdoor,70


Après nettoyage, l’entraînement compte 101 269 flux (48,8 % d’attaques) et le test 52 648 flux (36,05 % d’attaques). Le nettoyage a donc inversé la tendance des fichiers bruts, où les attaques étaient majoritaires (68 %) : ces fichiers étaient fortement gonflés par des doublons, surtout pour la famille Generic. Les références du hasard pour la PR-AUC sont environ 0,49 (validation) et 0,36 (test).

Deux limites en découlent : les proportions de classes diffèrent entre entraînement et test, ce qui peut décaler les scores et le seuil optimal ; et les familles rares sont peu représentées dans le test (Worms 43 flux, Backdoor 70), si bien que leur rappel sera peu fiable.